In [1]:
from pathlib import Path
import re
import hashlib
import json
import unicodedata
import html
from collections import Counter

import pandas as pd

In [2]:
# Project root
PROJECT_ROOT = Path("../")

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FILTERED_DIR = PROJECT_ROOT / "data" / "filtered"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

# Create directories if they don't exist
for directory in [RAW_DIR, CLEANED_DIR, FILTERED_DIR, FINAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw:", RAW_DIR.resolve())
print("Cleaned:", CLEANED_DIR.resolve())
print("Filtered:", FILTERED_DIR.resolve())
print("Final:", FINAL_DIR.resolve())

Project root: /
Raw: /data/raw
Cleaned: /data/cleaned
Filtered: /data/filtered
Final: /data/final


In [3]:
import sys
import subprocess

python = sys.executable

subprocess.run([
    python, "-m", "pip", "uninstall", "-y", "pyarrow"
])

subprocess.run([
    python, "-m", "pip", "install",
    "--no-cache-dir",
    "--force-reinstall",
    "pyarrow==21.0.0"
])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'pyarrow==21.0.0'], returncode=0)

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    
)

print(dataset)    

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [3]:
print(dataset.column_names)
print(dataset[0])

['text']
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [4]:
print(dataset.column_names)
print(dataset[2])

['text']
{'text': 'One day, a little fish named Fin was swimming near the shore. He saw a big crab and wanted to be friends. "Hi, I am Fin. Do you want to play?" asked the little fish. The crab looked at Fin and said, "No, I don\'t want to play. I am cold and I don\'t feel fine."\n\nFin felt sad but wanted to help the crab feel better. He swam away and thought of a plan. He remembered that the sun could make things warm. So, Fin swam to the top of the water and called to the sun, "Please, sun, help my new friend feel fine and not freeze!"\n\nThe sun heard Fin\'s call and shone its warm light on the shore. The crab started to feel better and not so cold. He saw Fin and said, "Thank you, little fish, for making me feel fine. I don\'t feel like I will freeze now. Let\'s play together!" And so, Fin and the crab played and became good friends.'}


In [5]:
len(dataset)

2119719

In [7]:
from pathlib import Path
import json

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

raw_file = RAW_DIR / "tinystories_train.jsonl"

with open(raw_file, "w", encoding="utf-8") as f:
    for item in dataset:
        record = {
            "text": item["text"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved raw data to: {raw_file}")
print(f"File size: {raw_file.stat().st_size / (1024**2):.2f} MB")

Saved raw data to: data/raw/tinystories_train.jsonl
File size: 1868.20 MB


# Row data statistics

In [8]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return records


raw_docs = load_jsonl(raw_file)

print("Number of raw documents:", len(raw_docs))

Number of raw documents: 2119719


In [9]:
raw_chars = sum(len(doc["text"]) for doc in raw_docs)

print("Raw documents:", len(raw_docs))
print("Raw characters:", raw_chars)
print("Raw size MB:", round(raw_file.stat().st_size / (1024**2), 2))

Raw documents: 2119719
Raw characters: 1899973203
Raw size MB: 1868.2


# Cleaning

# RAW
#  ↓
# CLEANING
#  ↓
# CLEANED

# HTML removal
# HTML entity decoding
# Unicode NFKC normalization
# whitespace normalization
# control-character removal
# empty-document removal

# Cleaning

# Class DataCleaner:

In [10]:
def remove_html(text):
    """
    Remove HTML/XML-like tags.
    """
    text = re.sub(r"<[^>]+>", " ", text)    
    return text


def normalize_unicode(text):
    """
    Normalize Unicode using NFKC.
    """
    return unicodedata.normalize("NFKC", text)


def remove_control_characters(text):
    """
    Remove control characters while keeping
    newline and tab.
    """
    return "".join(
        char for char in text
        if char in "\n\t" or not unicodedata.category(char).startswith("C")
    )


def normalize_whitespace(text):
    """
    Compress repeated whitespace.
    """
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def clean_text(text):
    """
    Complete cleaning pipeline.
    """
    text = html.unescape(text)
    text = remove_html(text)
    text = normalize_unicode(text)
    text = remove_control_characters(text)
    text = normalize_whitespace(text)

    return text

# Statitisktika uchun

In [12]:
import html
import re
import unicodedata

cleaned_docs = []

cleaning_stats = {
    "input": 0,
    "output": 0,
    "empty_removed": 0
}

for doc in raw_docs:

    cleaning_stats["input"] += 1

    original_text = doc["text"]
    cleaned_text = clean_text(original_text)

    if not cleaned_text:
        cleaning_stats["empty_removed"] += 1
        continue

    cleaned_docs.append({
        "text": cleaned_text
    })

cleaning_stats["output"] = len(cleaned_docs)

print(cleaning_stats)

{'input': 2119719, 'output': 2119489, 'empty_removed': 230}


# Tozalangan datani saqlash

In [13]:
from pathlib import Path
import json

BASE_DIR = Path(r"C:\Users\Rasulbekk\Desktop\my_LLM")

CLEANED_DIR = BASE_DIR / "data" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_file = CLEANED_DIR / "tinystories_cleaned.jsonl"

with open(cleaned_file, "w", encoding="utf-8") as f:
    for doc in cleaned_docs:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )

print(f"Saved cleaned data to: {cleaned_file}")

Saved cleaned data to: C:\Users\Rasulbekk\Desktop\my_LLM/data/cleaned/tinystories_cleaned.jsonl


# # filtering
# CLEANED
#    ↓
# FILTERING
#    ↓
# FILTERED

# #Minimum characters
# Maximum characters
# Alphabetic ratio
# Repeated-line ratio

In [14]:
FILTER_CONFIG = {
    "min_chars": 200,
    "max_chars": 100000,
    "min_alpha_ratio": 0.30,
    "max_repeated_line_ratio": 0.30
}

FILTER_CONFIG

{'min_chars': 200,
 'max_chars': 100000,
 'min_alpha_ratio': 0.3,
 'max_repeated_line_ratio': 0.3}

In [15]:
def alphabetic_ratio(text):
    """
    Percentage of characters that are alphabetic.
    """
    if not text:
        return 0.0

    alpha_count = sum(char.isalpha() for char in text)

    return alpha_count / len(text)


def repeated_line_ratio(text):
    """
    Ratio of repeated non-empty lines.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) <= 1:
        return 0.0

    counts = Counter(lines)

    repeated_lines = sum(
        count
        for count in counts.values()
        if count > 1
    )

    return repeated_lines / len(lines)

# Statistika uchun

In [17]:
from collections import Counter 

filtered_docs = []

filter_stats = {
    "input": len(cleaned_docs),
    "min_chars": 0,
    "max_chars": 0,
    "alpha_ratio": 0,
    "repeated_lines": 0,
    "output": 0
}

for doc in cleaned_docs:

    text = doc["text"]

    # Minimum length
    if len(text) < FILTER_CONFIG["min_chars"]:
        filter_stats["min_chars"] += 1
        continue

    # Maximum length
    if len(text) > FILTER_CONFIG["max_chars"]:
        filter_stats["max_chars"] += 1
        continue

    # Alphabetic ratio
    if alphabetic_ratio(text) < FILTER_CONFIG["min_alpha_ratio"]:
        filter_stats["alpha_ratio"] += 1
        continue

    # Repeated line ratio
    if repeated_line_ratio(text) > FILTER_CONFIG["max_repeated_line_ratio"]:
        filter_stats["repeated_lines"] += 1
        continue

    filtered_docs.append(doc)

filter_stats["output"] = len(filtered_docs)

print(filter_stats)

{'input': 2119489, 'min_chars': 50, 'max_chars': 0, 'alpha_ratio': 0, 'repeated_lines': 49, 'output': 2119390}


# Fiter bolgan datani saqlash

In [18]:
from pathlib import Path
import json

BASE_DIR = Path(r"C:\Users\Rasulbekk\Desktop\my_LLM")

FILTERED_DIR = BASE_DIR / "data" / "filtered"
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

filtered_file = FILTERED_DIR / "tinystories_filtered.jsonl"

with open(filtered_file, "w", encoding="utf-8") as f:
    for doc in filtered_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print(f"Saved filtered data to: {filtered_file}")

Saved filtered data to: C:\Users\Rasulbekk\Desktop\my_LLM/data/filtered/tinystories_filtered.jsonl


# #Deduplication
# FILTERED
#    ↓
# DEDUPLICATION
#    ↓
# UNIQUE DATA

# Normalize the text
# Calculate SHA-256
# Store hashes in a set
# Remove repeated documents

In [19]:
def document_hash(text):
    """
    Create SHA-256 hash for a document.
    """
    normalized = text.strip().lower()

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

In [21]:
import hashlib

unique_docs = []
seen_hashes = set()

duplicate_count = 0

for doc in filtered_docs:

    text = doc["text"]

    doc_hash = document_hash(text)

    if doc_hash in seen_hashes:
        duplicate_count += 1
        continue

    seen_hashes.add(doc_hash)
    unique_docs.append(doc)

print("Input documents:", len(filtered_docs))
print("Duplicates removed:", duplicate_count)
print("Unique documents:", len(unique_docs))

Input documents: 2119390
Duplicates removed: 320226
Unique documents: 1799164


# PII scrubbing

# UNIQUE DATA
    ↓
# PII SCRUBBING
    ↓
# PII-SAFE DATA

# Email
# Phone numbers
# Large digit sequences

In [22]:
EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)"
)

NUMBER_PATTERN = re.compile(
    r"(?<!\d)\d{6,}(?!\d)"
)

In [23]:
def scrub_pii(text):
    """
    Replace common PII patterns with placeholders.
    """

    stats = {
        "email": 0,
        "phone": 0,
        "digits": 0
    }

    text, stats["email"] = EMAIL_PATTERN.subn(
        "<EMAIL>",
        text
    )

    text, stats["phone"] = PHONE_PATTERN.subn(
        "<PHONE>",
        text
    )

    text, stats["digits"] = NUMBER_PATTERN.subn(
        "<NUMBER>",
        text
    )

    return text, stats

In [24]:
final_docs = []

pii_stats = {
    "email": 0,
    "phone": 0,
    "digits": 0
}

for doc in unique_docs:

    cleaned_text, stats = scrub_pii(
        doc["text"]
    )

    for key in pii_stats:
        pii_stats[key] += stats[key]

    final_docs.append({
        "text": cleaned_text
    })

print("PII matches:")
print(pii_stats)

PII matches:
{'email': 0, 'phone': 23, 'digits': 1}


# Save final corpus

In [25]:
from pathlib import Path
import json

BASE_DIR = Path(r"C:\Users\Rasulbekk\Desktop\my_LLM")

FINAL_DIR = BASE_DIR / "data" / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_file = FINAL_DIR / "tinystories_final.jsonl"

with open(final_file, "w", encoding="utf-8") as f:
    for doc in final_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print(f"Final corpus saved to: {final_file}")

Final corpus saved to: C:\Users\Rasulbekk\Desktop\my_LLM/data/final/tinystories_final.jsonl


our-llm/
└── data/
    ├── raw/
    │   └── tinystories_train.jsonl
    │
    ├── cleaned/
    │   └── tinystories_cleaned.jsonl
    │
    ├── filtered/
    │   └── tinystories_filtered.jsonl
    │
    └── final/
        └── tinystories_final.jsonl

# Final statistics( how many in how many out)

In [26]:
final_chars = sum(
    len(doc["text"])
    for doc in final_docs
)

stats_report = f"""# Data Pipeline Statistics

## Raw

- Documents: {len(raw_docs)}
- Characters: {raw_chars}
- Size: {raw_file.stat().st_size / (1024**2):.2f} MB

## Cleaning

- Input documents: {cleaning_stats["input"]}
- Output documents: {cleaning_stats["output"]}
- Empty/broken documents removed: {cleaning_stats["empty_removed"]}

## Filtering

- Input documents: {filter_stats["input"]}
- Removed by min_chars: {filter_stats["min_chars"]}
- Removed by max_chars: {filter_stats["max_chars"]}
- Removed by alphabetic ratio: {filter_stats["alpha_ratio"]}
- Removed by repeated-line ratio: {filter_stats["repeated_lines"]}
- Output documents: {filter_stats["output"]}

## Deduplication

- Input documents: {len(filtered_docs)}
- Exact duplicates removed: {duplicate_count}
- Unique documents: {len(unique_docs)}

## PII Scrubbing

- Emails masked: {pii_stats["email"]}
- Phone numbers masked: {pii_stats["phone"]}
- Large digit sequences masked: {pii_stats["digits"]}

## Final

- Documents: {len(final_docs)}
- Characters: {final_chars}

## Pipeline

Raw
→ Cleaning
→ Filtering
→ Deduplication
→ PII Scrubbing
→ Final Corpus
"""

report_file = FINAL_DIR / "stats_report.md"

with open(report_file, "w", encoding="utf-8") as f:
    f.write(stats_report)

print(stats_report)
print(f"Report saved to: {report_file}")

# Data Pipeline Statistics

## Raw

- Documents: 2119719
- Characters: 1899973203
- Size: 1868.20 MB

## Cleaning

- Input documents: 2119719
- Output documents: 2119489
- Empty/broken documents removed: 230

## Filtering

- Input documents: 2119489
- Removed by min_chars: 50
- Removed by max_chars: 0
- Removed by alphabetic ratio: 0
- Removed by repeated-line ratio: 49
- Output documents: 2119390

## Deduplication

- Input documents: 2119390
- Exact duplicates removed: 320226
- Unique documents: 1799164

## PII Scrubbing

- Emails masked: 0
- Phone numbers masked: 23
- Large digit sequences masked: 1

## Final

- Documents: 1799164
- Characters: 1602756918

## Pipeline

Raw
→ Cleaning
→ Filtering
→ Deduplication
→ PII Scrubbing
→ Final Corpus

Report saved to: C:\Users\Rasulbekk\Desktop\my_LLM/data/final/stats_report.md


In [27]:
stage_counts = {
    "Raw": len(raw_docs),
    "Cleaned": len(cleaned_docs),
    "Filtered": len(filtered_docs),
    "Deduplicated": len(unique_docs),
    "Final": len(final_docs)
}

for stage, count in stage_counts.items():
    retention = count / len(raw_docs) * 100

    print(
        f"{stage:15} : "
        f"{count:,} documents "
        f"({retention:.2f}% retained)"
    )

Raw             : 2,119,719 documents (100.00% retained)
Cleaned         : 2,119,489 documents (99.99% retained)
Filtered        : 2,119,390 documents (99.98% retained)
Deduplicated    : 1,799,164 documents (84.88% retained)
Final           : 1,799,164 documents (84.88% retained)


In [37]:
#befor and after cleaning
for i in range(min(5, len(raw_docs))):

    print("=" * 80)

    print("RAW:")
    print(raw_docs[i]["text"][:500])

    print("\nCLEANED:")
    print(cleaned_docs[i]["text"][:500])

RAW:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b

CLEANED:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not diff

In [38]:
for i, doc in enumerate(final_docs[:5]):

    print("=" * 80)
    print(f"FINAL DOCUMENT {i + 1}")
    print(doc["text"][:1000])

FINAL DOCUMENT 1
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
FINAL DOCUMENT 2
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree ha

                        
                 TODAY
                    ↓
┌────────────────────────────────────┐
│         DATA ENGINEERING           │
├────────────────────────────────────┤
│                                    │
│ Raw data                           │
│      ↓                             │
│ Cleaning                           │
│      ↓                             │
│ Filtering                          │
│      ↓                             │
│ Deduplication                      │
│      ↓                             │
│ PII Scrubbing                      │
│      ↓                             │
│ FINAL TRAINING CORPUS              │
│                                    │
└────────────────────────────────────┘
                    ↓
              NEXT LESSON
                    ↓
              Tokenizer
                    ↓
              Token IDs
                    ↓
            Dataset Builder
                    ↓
          train.bin / val.bin
                    ↓
               Training